# FINAL Step 2.7 — Tissue Preprocessing Methodology Checks

This notebook investigates five tissue-focused methodology checks prompted by supervisor/peer discussion:

1. raw-coordinate tissue patch extraction;
2. Otsu-selection bias audit;
3. masked/no-CLAHE tissue sensitivity;
4. unsupervised WS tissue sanity check;
5. frequency-domain tissue diagnostic.

It is intentionally self-contained and writes only to `data/outputs/tissue_preprocessing_methodology_checks/`. Existing notebooks are not edited or required.

In [1]:
from __future__ import annotations

import json
import math
import random
import sys
import warnings
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
from IPython.display import display
from kymatio.scattering2d.frontend.numpy_frontend import ScatteringNumPy2D as Scattering2D
from scipy.stats import wasserstein_distance
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score,
    adjusted_rand_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    silhouette_score,
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)


def find_project_root() -> Path:
    cur = Path.cwd().resolve()
    for cand in [cur, *cur.parents]:
        if (cand / "data").exists() and (cand / "Step 2 - experiments NOTEBOOKS").exists():
            return cand
    raise FileNotFoundError("Could not locate Project34 root")

PROJECT = find_project_root()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from project34.protocol import SEEDS, SubspaceKNN, set_seed

DATA = PROJECT / "data"
OUT = DATA / "outputs" / "tissue_preprocessing_methodology_checks"
OUT.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
LABEL_MAP = {"fatty": 0, "fibroglandular": 1}
VARIANTS = ["current_full_preprocessed", "raw_minmax", "raw_percentile", "masked_no_clahe"]

print("Project:", PROJECT)
print("Output:", OUT)
print("Seeds:", SEEDS)

Project: /home/nabeel/project34/Project34
Output: /home/nabeel/project34/Project34/data/outputs/tissue_preprocessing_methodology_checks
Seeds: [1, 7, 34, 42, 99]


In [2]:
def resolve_project_path(path_like) -> Path:
    """Resolve paths stored as ../data/... or absolute paths."""
    s = str(path_like).replace("\\", "/")
    p = Path(s)
    if p.is_absolute():
        return p
    marker = "data/"
    idx = s.find(marker)
    if idx >= 0:
        return PROJECT / s[idx:]
    return (PROJECT / p).resolve()


def resize_to_224(patch: np.ndarray) -> np.ndarray:
    return cv2.resize(patch.astype(np.float32), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA).astype(np.float32)


def minmax01(img: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32)
    lo, hi = float(np.nanmin(img)), float(np.nanmax(img))
    if hi <= lo:
        return np.zeros_like(img, dtype=np.float32)
    return ((img - lo) / (hi - lo)).clip(0, 1).astype(np.float32)


def percentile01(img: np.ndarray, lo_pct=1, hi_pct=99) -> np.ndarray:
    img = img.astype(np.float32)
    lo, hi = np.percentile(img, [lo_pct, hi_pct])
    if hi <= lo:
        return minmax01(img)
    return ((img - lo) / (hi - lo)).clip(0, 1).astype(np.float32)


_raw_dicom_cache = {}
_minmax_cache = {}
_percentile_cache = {}
_mask_cache = {}
_final_cache = {}


def load_dicom_pixels(path_like) -> np.ndarray:
    path = resolve_project_path(path_like)
    key = str(path)
    if key not in _raw_dicom_cache:
        ds = pydicom.dcmread(str(path))
        _raw_dicom_cache[key] = ds.pixel_array.astype(np.float32)
    return _raw_dicom_cache[key]


def cached_raw_minmax(path_like) -> np.ndarray:
    path = resolve_project_path(path_like)
    key = str(path)
    if key not in _minmax_cache:
        _minmax_cache[key] = minmax01(load_dicom_pixels(path))
    return _minmax_cache[key]


def cached_raw_percentile(path_like) -> np.ndarray:
    path = resolve_project_path(path_like)
    key = str(path)
    if key not in _percentile_cache:
        _percentile_cache[key] = percentile01(load_dicom_pixels(path))
    return _percentile_cache[key]


def load_npy_cached(path_like, cache) -> np.ndarray:
    path = resolve_project_path(path_like)
    key = str(path)
    if key not in cache:
        cache[key] = np.load(path).astype(np.float32)
    return cache[key]


def crop_from_coords(img: np.ndarray, row) -> np.ndarray:
    # Step 1.3 stores y1/x1 as inclusive endpoints.
    y0, y1, x0, x1 = int(row.y0), int(row.y1), int(row.x0), int(row.x1)
    return resize_to_224(img[y0:y1 + 1, x0:x1 + 1])


def source_image_for_row(row, variant: str, final_to_source: dict) -> np.ndarray:
    if variant == "current_full_preprocessed":
        return load_npy_cached(row.source_final_npy, _final_cache)
    source = final_to_source[str(resolve_project_path(row.source_final_npy))]
    if variant == "raw_minmax":
        return cached_raw_minmax(source["dicom"])
    if variant == "raw_percentile":
        return cached_raw_percentile(source["dicom"])
    if variant == "masked_no_clahe":
        img = cached_raw_percentile(source["dicom"]).copy()
        breast = load_npy_cached(source["breast_mask"], _mask_cache) > 0
        pect = load_npy_cached(source["pect_mask"], _mask_cache) > 0
        img[~breast] = 0.0
        img[pect] = 0.0
        return img
    raise ValueError(f"Unknown variant: {variant}")


def build_patch_array(manifest: pd.DataFrame, variant: str, final_to_source: dict) -> np.ndarray:
    cache_path = OUT / f"tissue_{variant}_patches_224.npy"
    if cache_path.exists():
        arr = np.load(cache_path)
        print(f"Loaded {variant} patches:", arr.shape)
        return arr
    patches = []
    for i, row in enumerate(manifest.itertuples(index=False), start=1):
        if variant == "current_full_preprocessed":
            patch = np.load(resolve_project_path(row.patch_npy)).astype(np.float32)
        else:
            patch = crop_from_coords(source_image_for_row(row, variant, final_to_source), row)
        patches.append(patch)
        if i % 100 == 0 or i == len(manifest):
            print(f"{variant}: extracted {i}/{len(manifest)}")
    arr = np.stack(patches).astype(np.float32)
    np.save(cache_path, arr)
    return arr


def ws_features(images: np.ndarray, variant: str, J=6, L=5, max_order=2) -> np.ndarray:
    cache_path = OUT / f"tissue_{variant}_ws_J{J}_L{L}_avg_features.npy"
    if cache_path.exists():
        feats = np.load(cache_path)
        print(f"Loaded {variant} WS features:", feats.shape)
        return feats
    scattering = Scattering2D(J=J, shape=(IMG_SIZE, IMG_SIZE), L=L, max_order=max_order)
    feats = []
    for i, img in enumerate(images, start=1):
        S = scattering(img.astype(np.float32))
        feats.append(S.reshape(S.shape[0], -1).mean(axis=1))
        if i % 50 == 0 or i == len(images):
            print(f"{variant}: WS {i}/{len(images)}")
    feats = np.stack(feats).astype(np.float32)
    np.save(cache_path, feats)
    return feats


def image_level_split(groups, acr_by_img, seed, test_size=0.2):
    meta = acr_by_img.reset_index()
    _, test_files = train_test_split(meta["file_id"], test_size=test_size, stratify=meta["acr"], random_state=seed, shuffle=True)
    is_test = pd.Series(groups).isin(set(test_files)).to_numpy()
    return np.where(~is_test)[0], np.where(is_test)[0]


def evaluate_ws_features(X, y, groups, acr_by_img, variant: str) -> tuple[pd.DataFrame, dict]:
    rows = []
    for seed in SEEDS:
        tr, te = image_level_split(groups, acr_by_img, seed)
        pipe = Pipeline([("sc", StandardScaler()), ("clf", SubspaceKNN(80, min(190, X.shape[1]), 1, seed))])
        pipe.fit(X[tr], y[tr])
        pred = pipe.predict(X[te])
        proba = pipe.predict_proba(X[te])[:, 1]
        rows.append({
            "variant": variant,
            "method": "WS-only (averaged)",
            "seed": seed,
            "n_test": len(te),
            "test_acc": accuracy_score(y[te], pred),
            "auroc": roc_auc_score(y[te], proba),
            "precision": precision_score(y[te], pred, zero_division=0),
            "recall": recall_score(y[te], pred, zero_division=0),
            "f1": f1_score(y[te], pred, zero_division=0),
        })
    per_seed = pd.DataFrame(rows)

    tr34, _ = image_level_split(groups, acr_by_img, 34)
    cv = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=34)
    pipe = Pipeline([("sc", StandardScaler()), ("clf", SubspaceKNN(80, min(190, X.shape[1]), 1, 34))])
    cv_scores = cross_val_score(pipe, X[tr34], y[tr34], groups=groups[tr34], cv=cv, scoring="accuracy", n_jobs=1)

    summary = {
        "variant": variant,
        "method": "WS-only (averaged)",
        "auroc": per_seed["auroc"].mean(),
        "auroc_sd": per_seed["auroc"].std(ddof=0),
        "test_acc": per_seed["test_acc"].mean(),
        "test_acc_sd": per_seed["test_acc"].std(ddof=0),
        "f1": per_seed["f1"].mean(),
        "cv_acc_seed34": float(cv_scores.mean()),
        "cv_sd_seed34": float(cv_scores.std()),
    }
    return per_seed, summary


def radial_power_spectrum(img: np.ndarray, n_bins=40) -> np.ndarray:
    arr = img.astype(np.float32) - float(np.mean(img))
    power = np.abs(np.fft.fftshift(np.fft.fft2(arr))) ** 2
    h, w = power.shape
    yy, xx = np.indices((h, w))
    rr = np.sqrt((yy - h / 2) ** 2 + (xx - w / 2) ** 2)
    bins = np.linspace(0, rr.max(), n_bins + 1)
    out = np.zeros(n_bins, dtype=np.float64)
    for i in range(n_bins):
        m = (rr >= bins[i]) & (rr < bins[i + 1])
        out[i] = power[m].mean() if np.any(m) else 0.0
    out = np.log1p(out)
    return out.astype(np.float32)


def patch_stats(images: np.ndarray, features: np.ndarray | None = None) -> pd.DataFrame:
    flat = images.reshape(len(images), -1)
    out = pd.DataFrame({
        "mean": flat.mean(axis=1),
        "std": flat.std(axis=1),
        "p05": np.percentile(flat, 5, axis=1),
        "p50": np.percentile(flat, 50, axis=1),
        "p95": np.percentile(flat, 95, axis=1),
    })
    spectra = np.stack([radial_power_spectrum(im) for im in images])
    out["fft_low"] = spectra[:, :8].mean(axis=1)
    out["fft_mid"] = spectra[:, 8:24].mean(axis=1)
    out["fft_high"] = spectra[:, 24:].mean(axis=1)
    if features is not None:
        out["ws_energy"] = np.linalg.norm(features, axis=1)
    return out


def sample_rejected_candidates(manifest: pd.DataFrame, final_to_source: dict, n_per_label=180, seed=34) -> pd.DataFrame:
    """Approximate rejected candidates: same images/sizes, valid breast/pect area, but low Otsu target purity."""
    rng = np.random.default_rng(seed)
    records = []
    by_label = {"fatty": 0, "fibroglandular": 0}
    max_trials = 120000
    rows = manifest.sample(frac=1, random_state=seed).reset_index(drop=True)
    trial = 0
    while min(by_label.values()) < n_per_label and trial < max_trials:
        trial += 1
        row = rows.iloc[int(rng.integers(0, len(rows)))]
        label = row.label
        if by_label[label] >= n_per_label:
            continue
        source = final_to_source[str(resolve_project_path(row.source_final_npy))]
        img = load_npy_cached(row.source_final_npy, _final_cache)
        breast = load_npy_cached(source["breast_mask"], _mask_cache) > 0
        pect = load_npy_cached(source["pect_mask"], _mask_cache) > 0
        allowed = breast & (~pect)
        s = int(row.orig_size)
        if img.shape[0] <= s or img.shape[1] <= s:
            continue
        y0 = int(rng.integers(0, img.shape[0] - s))
        x0 = int(rng.integers(0, img.shape[1] - s))
        y1, x1 = y0 + s, x0 + s
        patch_allowed = allowed[y0:y1, x0:x1]
        if patch_allowed.mean() < 0.90:
            continue
        vals = img[allowed]
        if vals.size < 10:
            continue
        thr = float(np.median(vals)) if np.allclose(vals.min(), vals.max()) else float(np.percentile(vals, 50))
        # Use the stored per-image Otsu threshold when available; it is how accepted rows were labelled.
        if not pd.isna(row.threshold):
            thr = float(row.threshold)
        patch = img[y0:y1, x0:x1].copy()
        if label == "fatty":
            target = (patch <= max(0.0, thr - 0.03)) & patch_allowed
        else:
            target = (patch >= min(1.0, thr + 0.03)) & patch_allowed
        target_frac = target.sum() / max(1, patch_allowed.sum())
        # Rejected because it would not satisfy target purity; keep it to audit selection bias.
        if target_frac >= 0.65:
            continue
        patch[~patch_allowed] = 0.0
        patch224 = resize_to_224(patch)
        records.append({
            "file_id": row.file_id,
            "label": label,
            "acr": row.acr,
            "orig_size": s,
            "y0": y0,
            "y1": y1 - 1,
            "x0": x0,
            "x1": x1 - 1,
            "allowed_frac": float(patch_allowed.mean()),
            "target_frac": float(target_frac),
            "mean_intensity": float(patch224.mean()),
            "std_intensity": float(patch224.std()),
        })
        npy = OUT / "rejected_candidate_patches" / label / f"{int(row.file_id)}_{label}_reject_{by_label[label]:03d}.npy"
        npy.parent.mkdir(parents=True, exist_ok=True)
        np.save(npy, patch224.astype(np.float32))
        records[-1]["patch_npy"] = str(npy.relative_to(PROJECT))
        by_label[label] += 1
    print("Rejected candidates sampled:", by_label, "trials:", trial)
    return pd.DataFrame(records)


def purity_against_labels(y_true, cluster_labels) -> float:
    score = 0
    for c in np.unique(cluster_labels):
        labs, counts = np.unique(y_true[cluster_labels == c], return_counts=True)
        score += counts.max()
    return score / len(y_true)


def summarize_mean_diff(df, group_col, value_cols):
    rows = []
    groups = list(df[group_col].dropna().unique())
    if len(groups) != 2:
        return pd.DataFrame()
    a, b = groups
    A = df[df[group_col] == a]
    B = df[df[group_col] == b]
    for c in value_cols:
        rows.append({
            "metric": c,
            f"mean_{a}": A[c].mean(),
            f"mean_{b}": B[c].mean(),
            "abs_diff": abs(A[c].mean() - B[c].mean()),
            "wasserstein": wasserstein_distance(A[c].dropna(), B[c].dropna()),
        })
    return pd.DataFrame(rows).sort_values("wasserstein", ascending=False)

## 1. Build Tissue Patch Variants

The current Step 1.3 tissue manifest provides the patch coordinates and labels. For this investigation, those coordinates are held fixed and only the pixel source changes:

- `current_full_preprocessed`: the existing saved tissue patch arrays;
- `raw_minmax`: original DICOM pixels with full-image min-max scaling;
- `raw_percentile`: original DICOM pixels with 1/99 percentile scaling only;
- `masked_no_clahe`: percentile-scaled DICOM pixels with breast and pectoral masks applied, but no CLAHE.

This isolates whether preprocessing changes the tissue WS representation without changing patch locations.

In [3]:
preproc = pd.read_csv(DATA / "outputs" / "preprocessed" / "preproc_index.csv")
preproc["final_resolved"] = preproc["final_npy"].map(lambda p: str(resolve_project_path(p)))
final_to_source = {
    row.final_resolved: {
        "dicom": str(resolve_project_path(row.dicom)),
        "breast_mask": str(resolve_project_path(row.breast_mask_npy)),
        "pect_mask": str(resolve_project_path(row.pect_mask_npy)),
    }
    for row in preproc.itertuples(index=False)
}

manifest = pd.read_csv(DATA / "outputs" / "background_patches" / "patches_index.csv")
manifest = manifest[manifest["label"].isin(LABEL_MAP)].copy().reset_index(drop=True)
manifest["patch_npy"] = manifest["patch_npy"].map(lambda p: str(resolve_project_path(p)))
manifest["source_final_npy"] = manifest["source_final_npy"].map(lambda p: str(resolve_project_path(p)))
manifest["class_id"] = manifest["label"].map(LABEL_MAP).astype(int)
manifest["file_id"] = manifest["file_id"].astype(str)
manifest.to_csv(OUT / "tissue_methodology_manifest.csv", index=False)

# ACR-stratified image-level split, matching FINAL Step 2.1.
y = manifest["class_id"].to_numpy(np.int64)
groups = manifest["file_id"].to_numpy()
acr_by_img = manifest.groupby("file_id")["acr"].first().astype(int)

patch_arrays = {}
ws_by_variant = {}
for variant in VARIANTS:
    patch_arrays[variant] = build_patch_array(manifest, variant, final_to_source)
    ws_by_variant[variant] = ws_features(patch_arrays[variant], variant)

print(f"{len(manifest)} tissue patches from {manifest['file_id'].nunique()} images")
print("Label counts:", manifest["label"].value_counts().to_dict())
print("ACR over images:", acr_by_img.value_counts().sort_index().to_dict())
display(manifest.head())

Loaded current_full_preprocessed patches: (434, 224, 224)
Loaded current_full_preprocessed WS features: (434, 406)
Loaded raw_minmax patches: (434, 224, 224)
Loaded raw_minmax WS features: (434, 406)
Loaded raw_percentile patches: (434, 224, 224)
Loaded raw_percentile WS features: (434, 406)
Loaded masked_no_clahe patches: (434, 224, 224)
Loaded masked_no_clahe WS features: (434, 406)
434 tissue patches from 100 images
Label counts: {'fatty': 251, 'fibroglandular': 183}
ACR over images: {1: 36, 2: 35, 3: 21, 4: 8}


,file_id,label,acr,patch_npy,patch_png,source_final_npy,source_breast_mask_npy,source_pect_mask_npy,xml_path,threshold,orig_size,y0,y1,x0,x1,allowed_frac,target_frac,mean_intensity,std_intensity,class_id
0,20586908,fatty,2,/home/nabeel/project34/Project34/data/outputs/...,../data/outputs/background_patches/fatty/20586...,/home/nabeel/project34/Project34/data/outputs/...,../data/outputs/preprocessed/breast_mask/20586...,../data/outputs/preprocessed/pect_mask/2058690...,../data/raw/kaggle_inbreast/AllXML/20586908.xml,0.681258,273,1135,1407,1804,2076,1.0,0.748849,0.571869,0.126487,0
1,20586908,fatty,2,/home/nabeel/project34/Project34/data/outputs/...,../data/outputs/background_patches/fatty/20586...,/home/nabeel/project34/Project34/data/outputs/...,../data/outputs/preprocessed/breast_mask/20586...,../data/outputs/preprocessed/pect_mask/2058690...,../data/raw/kaggle_inbreast/AllXML/20586908.xml,0.681258,351,1159,1509,2530,2880,1.0,0.658777,0.605455,0.102193,0
2,20586908,fatty,2,/home/nabeel/project34/Project34/data/outputs/...,../data/outputs/background_patches/fatty/20586...,/home/nabeel/project34/Project34/data/outputs/...,../data/outputs/preprocessed/breast_mask/20586...,../data/outputs/preprocessed/pect_mask/2058690...,../data/raw/kaggle_inbreast/AllXML/20586908.xml,0.681258,351,2149,2499,1915,2265,1.0,0.754263,0.579174,0.092580,0
3,20586908,fibroglandular,2,/home/nabeel/project34/Project34/data/outputs/...,../data/outputs/background_patches/fibroglandu...,/home/nabeel/project34/Project34/data/outputs/...,../data/outputs/preprocessed/breast_mask/20586...,../data/outputs/preprocessed/pect_mask/2058690...,../data/raw/kaggle_inbreast/AllXML/20586908.xml,0.681258,273,1446,1718,1853,2125,1.0,0.783936,0.846359,0.136011,1
4,20586908,fibroglandular,2,/home/nabeel/project34/Project34/data/outputs/...,../data/outputs/background_patches/fibroglandu...,/home/nabeel/project34/Project34/data/outputs/...,../data/outputs/preprocessed/breast_mask/20586...,../data/outputs/preprocessed/pect_mask/2058690...,../data/raw/kaggle_inbreast/AllXML/20586908.xml,0.681258,351,1797,2147,1862,2212,1.0,0.714726,0.804286,0.147768,1


## 2. Raw-Coordinate / No-CLAHE WS Evaluation

This section tests the raw-coordinate and no-CLAHE tissue variants with the paper-style averaged WS branch across the locked five image-level seeds. Full CNN retraining is not repeated here because it is expensive and would duplicate FINAL Step 2.1; the current CNN and fusion rows are included as baselines from the canonical tissue notebook.

In [ ]:
per_seed_frames = []
summary_rows = []
for variant in VARIANTS:
    per_seed, summary = evaluate_ws_features(ws_by_variant[variant], y, groups, acr_by_img, variant)
    per_seed_frames.append(per_seed)
    summary_rows.append(summary)

ws_per_seed = pd.concat(per_seed_frames, ignore_index=True)
ws_summary = pd.DataFrame(summary_rows)

# Attach Razali/current reference gaps for interpretability.
PAPER_WS_AUROC = 0.94
PAPER_WS_TEST = 0.914
ws_summary["paper_auroc"] = PAPER_WS_AUROC
ws_summary["paper_test_acc"] = PAPER_WS_TEST
ws_summary["auroc_abs_gap_to_paper"] = (ws_summary["auroc"] - PAPER_WS_AUROC).abs()
ws_summary["test_acc_abs_gap_to_paper"] = (ws_summary["test_acc"] - PAPER_WS_TEST).abs()
ws_summary = ws_summary.sort_values("auroc", ascending=False)

ws_per_seed.to_csv(OUT / "tissue_ws_pixel_source_per_seed.csv", index=False)
ws_summary.to_csv(OUT / "tissue_ws_pixel_source_summary.csv", index=False)

final_tissue = pd.read_csv(DATA / "outputs" / "final_tissue_replication" / "master_table_tissue.csv")
cnn_fusion_reference = final_tissue[final_tissue["Method"].isin(["CNN-only (ResNet18→kNN)", "CNN+WS fusion (concat)"])].copy()
cnn_fusion_reference.to_csv(OUT / "current_tissue_cnn_fusion_reference.csv", index=False)

print("WS pixel-source summary")
display(ws_summary.round(3))
print("Current CNN/fusion reference from FINAL Step 2.1 (not rerun here)")
display(cnn_fusion_reference)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_df = ws_summary.sort_values("auroc")
axes[0].barh(plot_df["variant"], plot_df["auroc"], xerr=plot_df["auroc_sd"], alpha=0.85)
axes[0].axvline(PAPER_WS_AUROC, color="black", linestyle="--", linewidth=1, label="Razali WS AUROC")
axes[0].set_xlabel("AUROC")
axes[0].set_title("Tissue WS by pixel source")
axes[0].legend(fontsize=8)
axes[0].grid(axis="x", alpha=0.3)
axes[1].barh(plot_df["variant"], plot_df["test_acc"], xerr=plot_df["test_acc_sd"], alpha=0.85, color="tab:orange")
axes[1].axvline(PAPER_WS_TEST, color="black", linestyle="--", linewidth=1, label="Razali WS test acc")
axes[1].set_xlabel("Test accuracy")
axes[1].set_title("Tissue WS by pixel source")
axes[1].legend(fontsize=8)
axes[1].grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "tissue_ws_pixel_source_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Otsu-Selection Bias Audit

The original sampler did not preserve every rejected trial, so this audit reconstructs an approximate rejected set from the same source images and patch-size distribution. A candidate is kept as `rejected_candidate` when it lies mostly in the allowed breast/non-pectoral area but fails the Otsu target-purity rule for the requested tissue class. We then compare accepted vs rejected candidates by intensity, Fourier energy, WS energy, and location.

In [ ]:
rejected_manifest_path = OUT / "rejected_candidate_manifest.csv"
if rejected_manifest_path.exists():
    rejected_manifest = pd.read_csv(rejected_manifest_path)
    print("Loaded rejected candidates:", rejected_manifest.shape)
else:
    rejected_manifest = sample_rejected_candidates(manifest, final_to_source, n_per_label=60, seed=34)
    rejected_manifest.to_csv(rejected_manifest_path, index=False)

rejected_patches_path = OUT / "rejected_candidate_patches_224.npy"
if rejected_patches_path.exists():
    rejected_patches = np.load(rejected_patches_path)
else:
    rejected_patches = np.stack([np.load(PROJECT / p).astype(np.float32) for p in rejected_manifest["patch_npy"]])
    np.save(rejected_patches_path, rejected_patches)

rejected_ws_path = OUT / "rejected_candidate_ws_J6_L5_avg_features.npy"
if rejected_ws_path.exists():
    rejected_ws = np.load(rejected_ws_path)
else:
    scattering = Scattering2D(J=6, shape=(IMG_SIZE, IMG_SIZE), L=5, max_order=2)
    rejected_ws = []
    for i, img in enumerate(rejected_patches, start=1):
        S = scattering(img.astype(np.float32))
        rejected_ws.append(S.reshape(S.shape[0], -1).mean(axis=1))
        if i % 50 == 0 or i == len(rejected_patches):
            print(f"rejected candidate WS {i}/{len(rejected_patches)}")
    rejected_ws = np.stack(rejected_ws).astype(np.float32)
    np.save(rejected_ws_path, rejected_ws)

accepted_stats = patch_stats(patch_arrays["current_full_preprocessed"], ws_by_variant["current_full_preprocessed"])
accepted_stats["selection"] = "accepted"
accepted_stats["label"] = manifest["label"].to_numpy()
accepted_stats["file_id"] = manifest["file_id"].to_numpy()
accepted_stats["x_center_norm"] = ((manifest["x0"] + manifest["x1"]) / 2) / manifest.groupby("file_id")["x1"].transform("max").replace(0, np.nan)
accepted_stats["y_center_norm"] = ((manifest["y0"] + manifest["y1"]) / 2) / manifest.groupby("file_id")["y1"].transform("max").replace(0, np.nan)
accepted_stats["orig_size"] = manifest["orig_size"].to_numpy()
accepted_stats["target_frac"] = manifest["target_frac"].to_numpy()
accepted_stats["allowed_frac"] = manifest["allowed_frac"].to_numpy()

rejected_stats = patch_stats(rejected_patches, rejected_ws)
rejected_stats["selection"] = "rejected_candidate"
rejected_stats["label"] = rejected_manifest["label"].to_numpy()
rejected_stats["file_id"] = rejected_manifest["file_id"].astype(str).to_numpy()
rejected_stats["x_center_norm"] = ((rejected_manifest["x0"] + rejected_manifest["x1"]) / 2) / rejected_manifest.groupby("file_id")["x1"].transform("max").replace(0, np.nan)
rejected_stats["y_center_norm"] = ((rejected_manifest["y0"] + rejected_manifest["y1"]) / 2) / rejected_manifest.groupby("file_id")["y1"].transform("max").replace(0, np.nan)
rejected_stats["orig_size"] = rejected_manifest["orig_size"].to_numpy()
rejected_stats["target_frac"] = rejected_manifest["target_frac"].to_numpy()
rejected_stats["allowed_frac"] = rejected_manifest["allowed_frac"].to_numpy()

audit_stats = pd.concat([accepted_stats, rejected_stats], ignore_index=True)
audit_stats.to_csv(OUT / "otsu_selection_bias_patch_stats.csv", index=False)

bias_metrics = summarize_mean_diff(
    audit_stats,
    "selection",
    ["mean", "std", "p05", "p50", "p95", "fft_low", "fft_mid", "fft_high", "ws_energy", "x_center_norm", "y_center_norm", "orig_size", "target_frac"],
)
bias_metrics.to_csv(OUT / "otsu_selection_bias_metric_differences.csv", index=False)

print("Accepted vs approximate rejected-candidate differences")
display(bias_metrics.round(4))

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, metric in zip(axes.ravel(), ["mean", "std", "fft_low", "fft_mid", "fft_high", "ws_energy"]):
    for selection, grp in audit_stats.groupby("selection"):
        ax.hist(grp[metric], bins=30, alpha=0.55, density=True, label=selection)
    ax.set_title(metric)
    ax.grid(alpha=0.25)
axes[0, 0].legend(fontsize=8)
plt.suptitle("Otsu-selection audit: accepted vs approximate rejected candidates")
plt.tight_layout()
plt.savefig(OUT / "otsu_selection_bias_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Unsupervised WS Tissue Sanity Check

This is not a replacement classifier. It asks whether the WS feature space naturally forms two clusters that align with the ACR-derived fatty/fibroglandular patch labels. We avoid PCA/t-SNE visual evidence here and instead report quantitative clustering metrics.

In [ ]:
cluster_rows = []
for variant, Xws in ws_by_variant.items():
    Z = StandardScaler().fit_transform(Xws)
    for seed in SEEDS:
        km = KMeans(n_clusters=2, random_state=seed, n_init=25)
        clusters = km.fit_predict(Z)
        cluster_rows.append({
            "variant": variant,
            "seed": seed,
            "purity": purity_against_labels(y, clusters),
            "adjusted_rand": adjusted_rand_score(y, clusters),
            "silhouette": silhouette_score(Z, clusters),
            "cluster0_n": int((clusters == 0).sum()),
            "cluster1_n": int((clusters == 1).sum()),
        })

cluster_per_seed = pd.DataFrame(cluster_rows)
cluster_summary = cluster_per_seed.groupby("variant").agg(
    purity=("purity", "mean"),
    purity_sd=("purity", "std"),
    adjusted_rand=("adjusted_rand", "mean"),
    adjusted_rand_sd=("adjusted_rand", "std"),
    silhouette=("silhouette", "mean"),
    silhouette_sd=("silhouette", "std"),
).reset_index().sort_values("adjusted_rand", ascending=False)

cluster_per_seed.to_csv(OUT / "unsupervised_ws_cluster_per_seed.csv", index=False)
cluster_summary.to_csv(OUT / "unsupervised_ws_cluster_summary.csv", index=False)

print("Unsupervised WS clustering summary")
display(cluster_summary.round(3))

fig, ax = plt.subplots(figsize=(8, 4))
plot = cluster_summary.sort_values("adjusted_rand")
ax.barh(plot["variant"], plot["adjusted_rand"], xerr=plot["adjusted_rand_sd"].fillna(0), alpha=0.85)
ax.set_xlabel("Adjusted Rand index vs ACR-derived patch labels")
ax.set_title("Unsupervised WS two-cluster sanity check")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "unsupervised_ws_cluster_ari.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Frequency-Domain Tissue Diagnostic

This diagnostic computes radially averaged Fourier power spectra for fatty and fibroglandular patches under each pixel-source variant. It directly addresses the concern that Otsu/background removal or contrast enhancement could change frequency information relevant to tissue separation.

In [ ]:
spectra = {}
freq_rows = []
for variant, imgs in patch_arrays.items():
    spec_path = OUT / f"tissue_{variant}_radial_fft_spectra.npy"
    if spec_path.exists():
        spec = np.load(spec_path)
    else:
        spec = np.stack([radial_power_spectrum(im, n_bins=40) for im in imgs])
        np.save(spec_path, spec)
    spectra[variant] = spec
    for label_name, class_id in LABEL_MAP.items():
        m = y == class_id
        mean_spec = spec[m].mean(axis=0)
        for bin_idx, val in enumerate(mean_spec):
            freq_rows.append({"variant": variant, "label": label_name, "freq_bin": bin_idx, "log_power": val})

freq_long = pd.DataFrame(freq_rows)
freq_long.to_csv(OUT / "tissue_radial_fft_mean_spectra_long.csv", index=False)

freq_sep_rows = []
for variant, spec in spectra.items():
    fatty = spec[y == LABEL_MAP["fatty"]]
    fibro = spec[y == LABEL_MAP["fibroglandular"]]
    diff = fibro.mean(axis=0) - fatty.mean(axis=0)
    freq_sep_rows.append({
        "variant": variant,
        "mean_abs_spectral_gap": float(np.mean(np.abs(diff))),
        "low_band_gap": float(np.mean(np.abs(diff[:8]))),
        "mid_band_gap": float(np.mean(np.abs(diff[8:24]))),
        "high_band_gap": float(np.mean(np.abs(diff[24:]))),
        "wasserstein_all_bins": float(np.mean([wasserstein_distance(fatty[:, i], fibro[:, i]) for i in range(spec.shape[1])])),
    })
freq_summary = pd.DataFrame(freq_sep_rows).sort_values("mean_abs_spectral_gap", ascending=False)
freq_summary.to_csv(OUT / "tissue_frequency_separation_summary.csv", index=False)

print("Frequency separation summary")
display(freq_summary.round(4))

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
for ax, variant in zip(axes.ravel(), VARIANTS):
    spec = spectra[variant]
    for label_name, class_id, color in [("fatty", 0, "tab:blue"), ("fibroglandular", 1, "tab:orange")]:
        m = y == class_id
        mean = spec[m].mean(axis=0)
        sd = spec[m].std(axis=0)
        x = np.arange(len(mean))
        ax.plot(x, mean, label=label_name, color=color)
        ax.fill_between(x, mean - sd, mean + sd, color=color, alpha=0.15)
    ax.set_title(variant)
    ax.grid(alpha=0.3)
axes[1, 0].set_xlabel("radial frequency bin")
axes[1, 1].set_xlabel("radial frequency bin")
axes[0, 0].set_ylabel("log radial power")
axes[1, 0].set_ylabel("log radial power")
axes[0, 0].legend(fontsize=8)
plt.suptitle("Fatty vs fibroglandular radial Fourier spectra by pixel source")
plt.tight_layout()
plt.savefig(OUT / "tissue_radial_fft_by_variant.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Synthesis and Recommendations

This final cell converts the numeric outputs into a compact recommendation table. The wording is deliberately cautious: these checks are methodology/sensitivity diagnostics, not replacements for the main paper-style replication.

In [ ]:
current_ws = ws_summary.loc[ws_summary["variant"] == "current_full_preprocessed"].iloc[0]
best_ws = ws_summary.sort_values("auroc", ascending=False).iloc[0]
masked_ws = ws_summary.loc[ws_summary["variant"] == "masked_no_clahe"].iloc[0]
raw_minmax_ws = ws_summary.loc[ws_summary["variant"] == "raw_minmax"].iloc[0]

largest_bias = bias_metrics.iloc[0]
best_cluster = cluster_summary.sort_values("adjusted_rand", ascending=False).iloc[0]
best_freq = freq_summary.sort_values("mean_abs_spectral_gap", ascending=False).iloc[0]

recommendations = [
    {
        "check": "1. Raw-coordinate tissue patch extraction",
        "headline": f"Best WS AUROC was {best_ws.variant} ({best_ws.auroc:.3f}); current preprocessing was {current_ws.auroc:.3f}.",
        "interpretation": "Raw-coordinate extraction is useful as a sensitivity test, but it does not automatically improve tissue WS.",
        "recommendation": "Keep as methodology sensitivity / appendix unless it materially beats the current row.",
    },
    {
        "check": "2. Otsu-selection bias audit",
        "headline": f"Largest accepted-vs-rejected shift was {largest_bias.metric} (Wasserstein {largest_bias.wasserstein:.3f}).",
        "interpretation": "The Otsu purity rule is not neutral if accepted and rejected candidates differ in intensity/frequency/WS-energy statistics.",
        "recommendation": "Keep this audit if the shifts are visible; it directly answers the supervisor's sampling-bias concern.",
    },
    {
        "check": "3. No-CLAHE tissue sensitivity",
        "headline": f"masked/no-CLAHE WS AUROC {masked_ws.auroc:.3f} vs current {current_ws.auroc:.3f}.",
        "interpretation": "No-CLAHE should only change the report narrative if it consistently improves or moves tissue closer to Razali.",
        "recommendation": "Use as a negative/control result if it is similar or worse; do not replace the main tissue pipeline on this alone.",
    },
    {
        "check": "4. Unsupervised WS tissue sanity check",
        "headline": f"Best ARI was {best_cluster.adjusted_rand:.3f} for {best_cluster.variant}; purity {best_cluster.purity:.3f}.",
        "interpretation": "This checks whether WS features naturally align with ACR-derived tissue labels without supervised fitting.",
        "recommendation": "Appendix/future-work unless ARI is clearly strong; do not treat it as a classifier replacement.",
    },
    {
        "check": "5. Frequency-domain tissue diagnostic",
        "headline": f"Largest fatty/fibro spectral gap was {best_freq.variant} (mean abs gap {best_freq.mean_abs_spectral_gap:.3f}).",
        "interpretation": "This is the most direct answer to the frequency-content concern, independent of classifier choice.",
        "recommendation": "Keep as a qualitative/diagnostic figure if it shows preprocessing changes spectral separation.",
    },
    {
        "check": "CNN/fusion feasibility",
        "headline": "Full CNN/fusion branches were not retrained for every pixel source in this notebook.",
        "interpretation": "Doing so would duplicate FINAL Step 2.1's expensive per-seed CNN training and is not necessary for the WS/frequency sampling-bias question.",
        "recommendation": "Only rerun CNN/fusion for a pixel source if WS/frequency diagnostics identify a clear candidate worth the compute.",
    },
]

recommendations_df = pd.DataFrame(recommendations)
recommendations_df.to_csv(OUT / "tissue_methodology_recommendations.csv", index=False)

with open(OUT / "tissue_methodology_summary.json", "w") as f:
    json.dump({
        "best_ws_variant": best_ws.to_dict(),
        "current_ws_variant": current_ws.to_dict(),
        "masked_no_clahe_variant": masked_ws.to_dict(),
        "largest_otsu_bias_metric": largest_bias.to_dict(),
        "best_unsupervised_cluster_variant": best_cluster.to_dict(),
        "best_frequency_variant": best_freq.to_dict(),
    }, f, indent=2)

print("Recommendations")
display(recommendations_df)
print("Saved outputs under", OUT)